# Damages With Road Assets (Illustrative Case)


This notebook reproduces the workflow from `examples/sito_2025_asset_based_damage_examples/illustrative_case.py`.

It builds a polygon from the hazard raster extent, configures network and hazard data, and runs an asset-based damages analysis.

## Imports

In [1]:
from pathlib import Path


import geopandas as gpd
import pyproj
import rasterio
from shapely.geometry import box as make_box
from shapely.ops import transform


from ra2ce.ra2ce_handler import Ra2ceHandler


from ra2ce.network.network_config_data.network_config_data import (
    HazardSection,
    NetworkConfigData,
    NetworkSection,
 )
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum


from ra2ce.analysis.damages.damages import AnalysisSectionDamages
from ra2ce.analysis.analysis_config_data.analysis_config_data import AnalysisConfigData
from ra2ce.analysis.analysis_config_data.enums.analysis_damages_enum import AnalysisDamagesEnum
from ra2ce.analysis.analysis_config_data.enums.damage_curve_enum import DamageCurveEnum
from ra2ce.analysis.analysis_config_data.enums.event_type_enum import EventTypeEnum
from ra2ce.analysis.analysis_config_data.enums.risk_calculation_mode_enum import RiskCalculationModeEnum

c:\Users\hauth\repositories\sito_2025_asset_based_damage_ra2ce\.pixi\envs\default\Lib\site-packages\rasterstats\io.py:17: FutureWarning: ReadingError is deprecated and will be removed in a future version. Use ShapelyError instead (functions previously raising {name} will now raise a ShapelyError instead).
  from shapely.errors import ReadingError
c:\Users\hauth\repositories\sito_2025_asset_based_damage_ra2ce\.pixi\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths And Input Data

In [2]:
root_dir = Path("data", "damage_assets")
static_path = root_dir.joinpath("static")
hazard_path = static_path.joinpath("hazard")
network_path = static_path.joinpath("network")
output_path = root_dir.joinpath("output")


## Configure Network And Hazard

In [ ]:
network_section = NetworkSection(
    network_type=NetworkTypeEnum.DRIVE,
    source=SourceEnum.OSM_DOWNLOAD,
    polygon=static_path / "polygon.geojson",
    save_gpkg=True,
    road_types=[
        RoadTypeEnum.TERTIARY,
        RoadTypeEnum.TERTIARY_LINK,
        RoadTypeEnum.SECONDARY,
        RoadTypeEnum.SECONDARY_LINK,
        RoadTypeEnum.PRIMARY,
        RoadTypeEnum.PRIMARY_LINK,
        RoadTypeEnum.TRUNK,
        RoadTypeEnum.MOTORWAY,
        RoadTypeEnum.MOTORWAY_LINK,
    ],
    reuse_network_output=True,
 )


hazard_section = HazardSection(
    hazard_map=[Path(file) for file in hazard_path.glob("*.tif")],
    aggregate_wl=AggregateWlEnum.MEAN,
    hazard_crs="EPSG:4326",
 )


network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=static_path,
    network=network_section,
    hazard=hazard_section,
 )
network_config_data.network.save_gpkg = True

NetworkConfigData(root_path=WindowsPath('data/damage_assets'), static_path=WindowsPath('data/damage_assets/static'), crs=<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich
, project=ProjectSection(name=''), network=NetworkSection(directed=False, source=<SourceEnum.OSM_DOWNLOAD: 2>, primary_file=None, diversion_file=None, file_id='', link_type_column='highway', polygon=WindowsPath('data/damage_assets/static/polygon.geojson'), network_type=<NetworkTypeEnum.DRIVE: 3>, road_types=[<RoadTypeEnum.TERTIARY: 9>, <RoadTypeEnum.TERTIARY_LINK: 10>, <RoadTypeEnum.SECONDARY: 7>, <RoadTypeEnum.SECONDARY_LINK: 8>, <RoadTypeEnum.PRIMARY: 5>, <RoadTypeEnum.PRIMARY_LINK: 6>, <RoadTypeEnum.TRUNK: 3>, <RoadTypeEnum.MOTORWAY: 1>, <RoadTypeEnum.MOTORWAY_

## Configure Damages Analysis

In [6]:
damages_analysis = [
    AnalysisSectionDamages(
        name="damages_with_asset",
        analysis=AnalysisDamagesEnum.DAMAGES_WITH_ASSETS,
        event_type=EventTypeEnum.RETURN_PERIOD,
        damage_curve=DamageCurveEnum.MAN,
        risk_calculation_mode=RiskCalculationModeEnum.TRIANGLE_TO_NULL_YEAR,
        risk_calculation_year=5,
        save_csv=True,
        save_gpkg=True,
    )
]


analysis_config_data = AnalysisConfigData(
    analyses=damages_analysis,
    root_path=root_dir,
    output_path=output_path,
 )
analysis_config_data.input_path = root_dir / "input_data"

## Run RA2CE

In [7]:
Ra2ceHandler.run_with_config_data(network_config_data, analysis_config_data)

c:\Users\hauth\repositories\sito_2025_asset_based_damage_ra2ce\.pixi\envs\default\Lib\site-packages\osmnx\simplification.py:596: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  merged = gdf_nodes.buffer(tolerance).union_all()
c:\Users\hauth\repositories\sito_2025_asset_based_damage_ra2ce\.pixi\envs\default\Lib\site-packages\osmnx\simplification.py:655: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = node_clusters.centroid
100%|██████████| 359/359 [00:00<00:00, 202713.40it/s]
2026-07-09 04:26:28 PM - [avg_speed_calculator.py:175] - root - WARNING - No valid file found with average speeds data\damage_assets\static\output_graph\avg_speed.csv, calculating and saving them instead.
2026-07-09 04:26:28 PM - [a

AttributeError: 'NoneType' object has no attribute 'columns'